# 2 Generating Ground Truth Data

## 2.1 Loading the documents

In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


## 2.2 Generating questions with structured output

In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from google.genai import Client
import os

load_dotenv()
google_client = Client(api_key=os.environ['GEMINI_API_KEY'])
MODEL_NAME = os.environ['MODEL_NAME']

In [10]:
import json
user_prompt = json.dumps(doc)

In [11]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [12]:
# messages = [
#     {"role": "developer", "content": data_gen_instructions},
#     {"role": "user", "content": user_prompt}
# ]

# Réécriture de la structure pour google
system_instruction = data_gen_instructions
user_content = user_prompt


In [13]:

import os
from google.genai import types
from dotenv import load_dotenv


load_dotenv()


response = google_client.models.generate_content(
    model=MODEL_NAME,
    contents=user_content,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json",
        response_schema=Questions
    )
)

In [14]:
# Vérifiez si la réponse est bien arrivée
print(f"La réponse est-elle None ? {response.parsed is None}")

# Si c'est None, regardez le texte brut (il contient souvent l'explication de l'erreur)
if response.parsed is None:
    print("Texte brut retourné par le modèle :")
    print(response.text)

La réponse est-elle None ? False


In [15]:
# response.output_parsed.questions
response.parsed.questions

['Is it too late for me to jump into the course if I just found out about it?',
 'Can I still sign up for the classes even though they started a while ago?',
 'If I start the course late, am I still allowed to participate?',
 'What happens if I join now but I want to get the certificate later?',
 "Do I need to follow a specific timeline to get certified if I'm joining the course today?"]

In [16]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

## 2.3 Reusable utilities

In [17]:
from evaluation_utils import llm_structured

In [18]:
# 1. Importez spécifiquement l'objet types dans l'espace de noms global du notebook
from google.genai import types 

# 2. Forcez la réimportation propre du module de fonctions
import evaluation_utils
import importlib
importlib.reload(evaluation_utils)

# 3. Testez maintenant l'appel
result, usage = evaluation_utils.llm_structured(
    google_client,
    data_gen_instructions,
    user_prompt,
    Questions # assurez-vous que cette variable est définie ici
)

In [19]:
# Vérifiez si le résultat est bien un objet (et non None)
print(type(result))
print(result)

# Si c'est un objet Pydantic, vous pouvez accéder à ses champs :
print(result.questions)

<class '__main__.Questions'>
questions=['Is it too late to register if I just found out about this course?', 'Can I catch up and complete the work for a certificate if I join late?', 'Do you still let new students sign up after the start date?', 'What are the requirements for getting certified if I start the course behind schedule?', 'Is there a cutoff point for submitting projects if I want to earn the certificate?']
['Is it too late to register if I just found out about this course?', 'Can I catch up and complete the work for a certificate if I join late?', 'Do you still let new students sign up after the start date?', 'What are the requirements for getting certified if I start the course behind schedule?', 'Is there a cutoff point for submitting projects if I want to earn the certificate?']


In [20]:
usage

## 2.4 Tracking cost

In [21]:
from evaluation_utils import calc_price

In [22]:
calc_price(usage)

{'input_cost': 0.0001305, 'output_cost': 0.000459, 'total_cost': 0.0005895}

In [23]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it too late to register if I just found out about this course?',
  'document': '74eb249bbf'},
 {'question': 'Can I catch up and complete the work for a certificate if I join late?',
  'document': '74eb249bbf'},
 {'question': 'Do you still let new students sign up after the start date?',
  'document': '74eb249bbf'},
 {'question': 'What are the requirements for getting certified if I start the course behind schedule?',
  'document': '74eb249bbf'},
 {'question': 'Is there a cutoff point for submitting projects if I want to earn the certificate?',
  'document': '74eb249bbf'}]

In [24]:
import pandas as pd

In [25]:
pd.DataFrame(records)

,question,document
0,Is it too late to register if I just found out...,74eb249bbf
1,Can I catch up and complete the work for a cer...,74eb249bbf
2,Do you still let new students sign up after th...,74eb249bbf
3,What are the requirements for getting certifie...,74eb249bbf
4,Is there a cutoff point for submitting project...,74eb249bbf


# 3 Generating Ground Truth for All Documents

In [26]:
from evaluation_utils import llm_structured_retry

In [27]:
import time
import json

def generate_ground_truth(doc):
    # Ajout d'une pause pour rester sous les 15 RPM 
    # (limites RPM de gemini 3.1 flash lite en free tier)
    # 60 secondes / 15 RPM = 4 secondes de sécurité par appel
    time.sleep(4) 

    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        google_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [28]:
generate_ground_truth(doc)

([{'question': 'Is it too late to jump into the course if I just found out about it today?',
   'document': '74eb249bbf'},
  {'question': "Can I still sign up for the program if I'm joining a bit late?",
   'document': '74eb249bbf'},
  {'question': 'Will I be able to get a certificate if I start the course right now?',
   'document': '74eb249bbf'},
  {'question': 'Are there any penalties for joining the class after it has already started?',
   'document': '74eb249bbf'},
  {'question': "What do I need to do to qualify for the certificate if I'm enrolling late?",
   'document': '74eb249bbf'}],
 <evaluation_utils.DummyUsage at 0x70fd8be74b90>)

In [29]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

## 3.1 Parallel processing

In [30]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [31]:
with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [32]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [33]:
ground_truth[10]

{'question': 'Where can I find the link to join the live workshop sessions?',
 'document': '489dd1c9d9'}

In [34]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08054025000000004

In [35]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08054025000000004

In [36]:
df_ground_truth = pd.DataFrame(ground_truth)

In [37]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [38]:
len(df_ground_truth)

565